# pavia4 METAL target (cls 5, theta=.075) — full cell suite, noise-before
Protocol (08-12): **gradient clip 1.0 only — no eigen floors, no MAD floors, no statistic guards.**
Cells: DARTS+robust rho sweep -> results table -> baselines (AMF/Levin/LRao-ES) -> LRao no-ES -> DART+ZCA -> download.

In [ ]:
!git clone -b rebuttal --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, torch
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
assert os.path.exists('repro/data/pavia-u.mat'), 'missing data'

In [ ]:
%%writefile run_darts_metal.py
"""DARTS + ROBUST front (median/1.4826*MAD) on PAVIA4 — METAL target
(GT class 5), theta=.075, NOISE-BEFORE. Colab port of
camera_ready/diagnostics/noise_before/run_nb_darts_robust.py (08-12):
self-contained on the repo (branch rebuttal), CUDA-enabled.
Env: RHOS (csv), SEEDS (csv, default 42,43), EPOCHS (default 10000),
OUT (json path). Keys: pavia4metal_darts_robust_r{rho}_s{seed}."""
import json
import os
import sys
import time

sys.path.insert(0, os.getcwd())

import numpy as np
import torch
from tqdm import tqdm

from repro import scenes
from repro.scenes import pavia_protocol as PP
from repro.protocols.spatial import load_cfg
from repro.core.data import Whitening, plant_targets, extract_neighborhoods
from repro.core.metrics import auc_safe, dr_at_fpr
from repro.core.seeding import seed_all
from repro.models.darts.model import _NeighborDenoiser

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT_JSON = os.environ.get('OUT', 'results_darts_metal.json')
RHOS = [float(x) for x in os.environ.get('RHOS', '0.5,0.03').split(',')]
SEEDS = [int(x) for x in os.environ.get('SEEDS', '42,43').split(',')]
EPOCHS = int(os.environ.get('EPOCHS', 10000))
EVAL_START, EVAL_EVERY = 200, 200
THETA = 0.075
TARGET_CLS = 5          # painted metal sheets

SP_CFG = load_cfg()
_SCENE = {}


def get_scene():
    if not _SCENE:
        sc = scenes.build('pavia4', SP_CFG)
        k = int(SP_CFG['k'])
        for key, flat, shape in (('_tr_nbr', sc['tr'], sc['tr_shape']),
                                 ('_te_nbr', sc['te'], sc['te_shape'])):
            img = torch.tensor(np.asarray(flat, np.float32)
                               .reshape(*shape, -1))
            _, nbr = extract_neighborhoods(img, k)
            sc[key] = nbr.numpy()
        _SCENE.update(sc)
    return _SCENE


def robust_front(tr):
    X = np.asarray(tr, np.float64)
    med = np.median(X, axis=0)
    mad = np.median(np.abs(X - med), axis=0) * 1.4826
    scale = mad   # no non-gradient clipping (user 08-12)
    return Whitening(med.astype(np.float32),
                     np.diag(1.0 / scale).astype(np.float32))


def run_one(rho, seed):
    key = f'pavia4metal_darts_robust_r{rho}_s{seed}'
    t0 = time.time()
    sc = get_scene()
    tr, te = sc['tr'], sc['te']
    s = PP.foreign_signature(sc['data'], sc['gt'], te,
                             cls=TARGET_CLS).astype(np.float32)
    D = tr.shape[1]
    W = robust_front(tr)
    cfg = dict(SP_CFG['darts'])
    sigma = float(np.sqrt(rho * float(np.mean(np.var(
        np.asarray(tr, np.float64), axis=0)))))
    seed_all(seed)
    net = _NeighborDenoiser(D, int(cfg['d_lat']), int(cfg['K']),
                            list(cfg['enc_hidden']),
                            list(cfg['score_hidden']),
                            float(np.sqrt(cfg['dsm_sigma_rho'])),
                            cfg['activation'], W).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=float(cfg['lr']),
                            weight_decay=float(cfg['weight_decay']))
    X = torch.tensor(np.asarray(tr, np.float32), device=DEVICE)
    N = torch.tensor(np.asarray(sc['_tr_nbr'], np.float32), device=DEVICE)
    planted, labels, _ = plant_targets(
        te, s, THETA, float(SP_CFG['target_fraction']), model='additive',
        seed=seed, spatial_shape=sc['te_shape'],
        edge_guard=int(SP_CFG['edge_guard']))
    planted = planted.astype(np.float32)
    y_lab = np.asarray(labels)
    gen = torch.Generator(device=DEVICE); gen.manual_seed(97 * seed)
    P, B = len(X), int(cfg['batch_size'])

    def evaluate():
        with torch.no_grad():
            def score_all(pix, nbr):
                out = []
                for i in range(0, len(pix), 1024):
                    p = torch.tensor(np.asarray(pix[i:i+1024], np.float32),
                                     device=DEVICE)
                    nb = torch.tensor(np.asarray(nbr[i:i+1024], np.float32),
                                      device=DEVICE)
                    out.append(net(p, nb).cpu().numpy())
                return np.concatenate(out, 0)
            z_tr = score_all(tr, sc['_tr_nbr'])
            z_te = score_all(planted, sc['_te_nbr'])
        zb = z_tr.mean(0)
        C = np.cov(z_tr, rowvar=False)
        T = -((z_te - zb) @ s) / np.sqrt(float(s @ C @ s))
        return auc_safe(y_lab, T), float(
            dr_at_fpr(y_lab, T, fpr_list=(0.05,))['0.05'])

    curve = []
    best = {'auc': -1.0}
    bar = tqdm(range(1, EPOCHS + 1), desc=key, ncols=130,
               mininterval=5.0, file=sys.stdout, ascii=True)
    for ep in bar:
        net.train()
        perm = torch.randperm(P, generator=gen, device=DEVICE)
        for i in range(0, P, B):
            sel = perm[i:i + B]
            eps = torch.randn((len(sel), D), generator=gen,
                              device=DEVICE) * sigma
            psi = net(X[sel] + eps, N[sel])
            loss = ((psi + eps / sigma ** 2) ** 2).sum(-1).mean()
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
        if ep >= EVAL_START and (ep % EVAL_EVERY == 0 or ep == EPOCHS):
            net.eval()
            auc, pd05 = evaluate()
            curve.append({'epoch': ep, 'auc': round(auc, 4),
                          'pd05': round(pd05, 4)})
            if auc > best['auc']:
                best = {'auc': auc, 'pd05': pd05, 'epoch': ep}
            bar.set_postfix_str(
                f'loss={float(loss):.3g} auc={auc:.3f} '
                f'best={best["auc"]:.3f}@{best["epoch"]}')
    bar.close()
    out = {'auc_best': round(best['auc'], 4),
           'pd05_best': round(best['pd05'], 4),
           'best_epoch': best['epoch'], 'theta': THETA,
           'target_cls': TARGET_CLS, 'final': curve[-1], 'rho': rho,
           'sigma_raw': round(sigma, 1), 'curve': curve,
           'device': DEVICE, 'sec': round(time.time() - t0)}
    res = json.load(open(OUT_JSON)) if os.path.exists(OUT_JSON) else {}
    res[key] = out
    json.dump(res, open(OUT_JSON, 'w'), indent=1)
    print(f'[{key}] best_auc={best["auc"]:.3f}@{best["epoch"]} '
          f'pd05={best["pd05"]:.3f} final={curve[-1]} ({out["sec"]}s)',
          flush=True)


if __name__ == '__main__':
    done = set(json.load(open(OUT_JSON)).keys()) if os.path.exists(OUT_JSON) else set()
    tasks = [(r, sd) for r in RHOS for sd in SEEDS
             if f'pavia4metal_darts_robust_r{r}_s{sd}' not in done]
    print(f'{len(tasks)} tasks, {EPOCHS} ep, device={DEVICE}', flush=True)
    get_scene()
    for r, sd in tasks:
        run_one(r, sd)
    print('ALL DONE', flush=True)


In [ ]:
!RHOS=0.03,1.0,2.0,0.001,0.05 SEEDS=42,43 EPOCHS=10000 python run_darts_metal.py

In [ ]:
import json
res = json.load(open('results_darts_metal.json'))
for k, v in sorted(res.items()):
    print(f"{k:45s} best={v['auc_best']:.3f}@{v['best_epoch']:<6} pd05={v['pd05_best']:.3f}")

In [ ]:
# Baselines on the METAL cell (cls 5, theta=.075): AMF-global/local, GMM-Levin, LRao(val-ES)
import json, os
import numpy as np, torch
from run_darts_metal import get_scene, SP_CFG, THETA, TARGET_CLS
from repro.scenes import pavia_protocol as PP
from repro.core.data import plant_targets
from repro.core.metrics import auc_safe, dr_at_fpr
from repro.protocols.spatial import _windows
from repro.models.classical import AMF, AMFLocal, GMMLevin
from repro.models.lrao import LRao

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEEDS = [42, 43]
sc = get_scene()
tr, te, shape = sc['tr'], sc['te'], sc['te_shape']
sig = PP.foreign_signature(sc['data'], sc['gt'], te, cls=TARGET_CLS).astype(np.float32)

amf = AMF(SP_CFG).fit(tr)
lev = GMMLevin(SP_CFG).fit(tr)
amf_local = AMFLocal(SP_CFG)
wA = amf_local.resolved_window(tr.shape[1])
_, nbr_amf = _windows(te, shape, wA, DEVICE)          # clean-image windows

lrao_cfg = dict(SP_CFG['lrao'])
w = (SP_CFG.get('net_width') or {}).get('pavia4')
if w: lrao_cfg['hidden'] = [int(w)]

res = {}
for seed in SEEDS:
    planted, labels, _ = plant_targets(
        te, sig, THETA, float(SP_CFG['target_fraction']), model='additive',
        seed=seed, spatial_shape=shape, edge_guard=int(SP_CFG['edge_guard']))
    planted = planted.astype(np.float32); y = np.asarray(labels)
    lrao = LRao(lrao_cfg).fit(tr, seed, DEVICE, run_dir=f'lrao_metal_s{seed}')
    scores = {
        'AMF-global': amf.score(planted, sig),
        'AMF-local':  amf_local.score(planted, nbr_amf, sig, device=DEVICE),
        'GMM-Levin':  lev.score(planted, sig),
        'LRao':       lrao.score(planted, tr[lrao.fit_idx], sig),
    }
    for name, T in scores.items():
        auc = float(auc_safe(y, T)); pd05 = float(dr_at_fpr(y, T, fpr_list=(0.05,))['0.05'])
        res.setdefault(name, {})[f's{seed}'] = {'auc': round(auc, 4), 'pd05': round(pd05, 4)}
        print(f'[{name} s{seed}] auc={auc:.3f} pd05={pd05:.3f}', flush=True)

for name, d in res.items():
    d['auc_mean'] = round(float(np.mean([v['auc'] for v in d.values()])), 4)
json.dump(res, open('results_baselines_metal.json', 'w'), indent=1)
print(json.dumps(res, indent=1))

In [ ]:
# LRao WITHOUT early stopping: full budget, no val selection, detection eval every 50 ep
import copy, json
import numpy as np, torch
from tqdm import tqdm
from run_darts_metal import get_scene, SP_CFG, THETA, TARGET_CLS
from repro.scenes import pavia_protocol as PP
from repro.core.data import plant_targets
from repro.core.metrics import auc_safe, dr_at_fpr
from repro.core.seeding import seed_all
from repro.models.lrao import LRao

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEEDS = [42, 43]
EPOCHS = 1000
EVAL_EVERY = 50

sc = get_scene()
tr_raw, te, shape = sc['tr'], sc['te'], sc['te_shape']
sig = PP.foreign_signature(sc['data'], sc['gt'], te, cls=TARGET_CLS).astype(np.float32)
cfg = dict(SP_CFG['lrao'])
w = (SP_CFG.get('net_width') or {}).get('pavia4')
if w: cfg['hidden'] = [int(w)]

res = {}
for seed in SEEDS:
    tr = np.asarray(tr_raw, np.float32)
    idx = np.random.default_rng(seed).permutation(len(tr))   # same split as fit()
    nv = max(1, int(len(tr) * float(cfg['val_fraction'])))
    fit_idx = idx[nv:]
    lr = LRao(cfg)
    lr.fit_idx = fit_idx
    seed_all(seed)                                           # BEFORE construction, as in fit()
    med, inv_scale = lr._robust_norm(tr[fit_idx])
    lr.med = torch.tensor(med, device=DEVICE)
    lr.inv_scale = torch.tensor(inv_scale, device=DEVICE)
    lr.net = lr._build_net(tr.shape[1]).to(DEVICE)
    opt = torch.optim.Adam(lr.net.parameters(), lr=float(cfg['lr']),
                           weight_decay=float(cfg['weight_decay']))
    Xf = torch.tensor(tr[fit_idx], device=DEVICE)
    P, B = len(Xf), int(cfg['batch_size'])

    planted, labels, _ = plant_targets(
        te, sig, THETA, float(SP_CFG['target_fraction']), model='additive',
        seed=seed, spatial_shape=shape, edge_guard=int(SP_CFG['edge_guard']))
    planted = planted.astype(np.float32); y = np.asarray(labels)

    curve, best = [], {'auc': -1.0}
    bar = tqdm(range(1, EPOCHS + 1), desc=f'LRao-noES s{seed}', dynamic_ncols=True)
    for ep in bar:
        lr.net.train()
        perm = torch.randperm(P, device=DEVICE)
        for i in range(0, P, B):
            loss = lr._lfi_cost(Xf[perm[i:i + B]])
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(lr.net.parameters(), float(cfg['grad_clip']))
            opt.step()
        if ep % EVAL_EVERY == 0 or ep == EPOCHS:
            lr.net.eval()
            T = lr.score(planted, tr[fit_idx], sig)
            auc = float(auc_safe(y, T))
            pd05 = float(dr_at_fpr(y, T, fpr_list=(0.05,))['0.05'])
            curve.append({'epoch': ep, 'auc': round(auc, 4), 'pd05': round(pd05, 4)})
            if auc > best['auc']:
                best = {'auc': auc, 'pd05': pd05, 'epoch': ep}
            bar.set_postfix_str(f'auc={auc:.3f} best={best["auc"]:.3f}@{best["epoch"]}')
    res[f's{seed}'] = {'auc_best': round(best['auc'], 4), 'pd05_best': round(best['pd05'], 4),
                       'best_epoch': best['epoch'], 'final': curve[-1], 'curve': curve}
    print(f"[LRao-noES s{seed}] best={best['auc']:.3f}@{best['epoch']} "
          f"pd05={best['pd05']:.3f} final={curve[-1]}", flush=True)

json.dump(res, open('results_lrao_noes_metal.json', 'w'), indent=1)

In [ ]:
# DART + ZCA front (NO eigen floor) on the METAL cell, noise-before, grad-clip 1.0
import json
import numpy as np, torch
from tqdm import tqdm
from run_darts_metal import get_scene, SP_CFG, THETA, TARGET_CLS
from repro.scenes import pavia_protocol as PP
from repro.core.data import Whitening, plant_targets
from repro.core.metrics import auc_safe, dr_at_fpr
from repro.core.models import ScoreNet

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RHOS = [0.003, 0.01, 0.03]
SEEDS = [42, 43]
EPOCHS = 10000
EVAL_START, EVAL_EVERY = 200, 100

sc = get_scene()
tr, te, shape = sc['tr'], sc['te'], sc['te_shape']
sig = PP.foreign_signature(sc['data'], sc['gt'], te, cls=TARGET_CLS).astype(np.float32)
D = tr.shape[1]

X64 = np.asarray(tr, np.float64)
mu = X64.mean(0)
C = np.cov(X64, rowvar=False)
lam, V = np.linalg.eigh(C)          # no floor - raw spectrum
Wz = (V @ np.diag(1.0 / np.sqrt(lam)) @ V.T)
W = Whitening(mu.astype(np.float32), Wz.astype(np.float32))
sigma_of = lambda rho: float(np.sqrt(rho * X64.var(0).mean()))

res = {}
for rho in RHOS:
    for seed in SEEDS:
        key = f'zca_r{rho}_s{seed}'
        sigma = sigma_of(rho)
        torch.manual_seed(seed)
        net = ScoreNet(D, [128], 'relu', whitening=W).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=5e-4)
        gen = torch.Generator(device=DEVICE); gen.manual_seed(97 * seed)
        X = torch.tensor(np.asarray(tr, np.float32), device=DEVICE)
        planted, labels, _ = plant_targets(
            te, sig, THETA, float(SP_CFG['target_fraction']), model='additive',
            seed=seed, spatial_shape=shape, edge_guard=int(SP_CFG['edge_guard']))
        Pt = torch.tensor(planted.astype(np.float32), device=DEVICE)
        y = np.asarray(labels)
        n = len(X)

        def psi(A):
            out = []
            with torch.no_grad():
                for i in range(0, len(A), 2048):
                    out.append(net(A[i:i+2048]).cpu().numpy())
            return np.concatenate(out, 0)

        best, curve = {'auc': -1.0}, []
        bar = tqdm(range(1, EPOCHS + 1), desc=key, dynamic_ncols=True,
                   mininterval=5.0, ascii=True)
        for ep in bar:
            net.train()
            perm = torch.randperm(n, generator=gen, device=DEVICE)
            for i in range(0, n, 512):
                b = X[perm[i:i+512]]
                eps = torch.randn(b.shape, generator=gen, device=DEVICE) * sigma
                loss = ((net(b + eps) + eps / sigma**2)**2).sum(-1).mean()
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                opt.step()
            if ep >= EVAL_START and (ep % EVAL_EVERY == 0 or ep == EPOCHS):
                net.eval()
                z_tr, z_te = psi(X), psi(Pt)
                zb = z_tr.mean(0); Cz = np.cov(z_tr, rowvar=False)
                T = -((z_te - zb) @ sig) / np.sqrt(float(sig @ Cz @ sig))
                auc = float(auc_safe(y, T))
                pd05 = float(dr_at_fpr(y, T, fpr_list=(0.05,))['0.05'])
                curve.append({'epoch': ep, 'auc': round(auc, 4), 'pd05': round(pd05, 4)})
                if auc > best['auc']:
                    best = {'auc': auc, 'pd05': pd05, 'epoch': ep}
                bar.set_postfix_str(f'loss={float(loss):.3g} auc={auc:.3f} '
                                    f'best={best["auc"]:.3f}@{best["epoch"]}')
        bar.close()
        res[key] = {'auc_best': round(best['auc'], 4), 'pd05_best': round(best['pd05'], 4),
                    'best_epoch': best['epoch'], 'final': curve[-1], 'curve': curve}
        print(f"[{key}] best={best['auc']:.3f}@{best['epoch']} pd05={best['pd05']:.3f} "
              f"final={curve[-1]}", flush=True)

json.dump(res, open('results_dart_zca_metal.json', 'w'), indent=1)

In [ ]:
from google.colab import files
import os
for f in ('results_darts_metal.json', 'results_baselines_metal.json',
          'results_lrao_noes_metal.json', 'results_dart_zca_metal.json'):
    if os.path.exists(f):
        files.download(f)